# Dataset Overview
We use the Twitter Entity Sentiment Analysis dataset from Kaggle.

Each row contains:

A tweet

The sentiment label

The topic or entity (like a game or product)

In [1]:
import kagglehub
import pandas as pd
import zipfile
import os

# Step 1: Download dataset
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
print("Dataset path:", path)

# Step 2: Find the zip file (auto-detect)
zip_files = [f for f in os.listdir(path) if f.endswith(".zip")]

# Step 3: Extract zip
for zip_file in zip_files:
    zip_path = os.path.join(path, zip_file)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(path)

print("Extraction done")

# Step 4: Load CSV files safely
train_path = os.path.join(path, "twitter_training.csv")
test_path = os.path.join(path, "twitter_validation.csv")

train_df = pd.read_csv(
    train_path,
    encoding="latin1",
    engine="python",
    on_bad_lines="skip"
)

test_df = pd.read_csv(
    test_path,
    encoding="latin1",
    engine="python",
    on_bad_lines="skip"
)

# Step 5: Assign column names (if needed)
train_df.columns = ["tweet_id", "entity", "sentiment", "tweet"]
test_df.columns = ["tweet_id", "entity", "sentiment", "tweet"]

# Step 6: Verify
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print(train_df.head())

Dataset path: C:\Users\FAUZAN\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2
Extraction done
Train shape: (74681, 4)
Test shape: (999, 4)
   tweet_id       entity sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                               tweet  
0  I am coming to the borders and I will kill you...  
1  im getting on borderlands and i will kill you ...  
2  im coming on borderlands and i will murder you...  
3  im getting on borderlands 2 and i will murder ...  
4  im getting into borderlands and i can murder y...  


The dataset files do not contain column names, which is common in real-world data.

So now we need to assign meaningful column names.

In [2]:
train_df.columns = ["tweet_id", "entity", "sentiment", "tweet"]
test_df.columns = ["tweet_id", "entity", "sentiment", "tweet"]

# Cleaning the Text
Machine learning models do not understand raw text.

Before converting text into numbers, we clean it slightly.

In [3]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"@\w+", "", text)           # remove mentions
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # remove special chars
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    return text

train_df["clean_tweet"] = train_df["tweet"].apply(clean_text)
test_df["clean_tweet"] = test_df["tweet"].apply(clean_text)

# Preparing Features and Labels

In [4]:
X_train = train_df["clean_tweet"]
y_train = train_df["sentiment"]

X_test = test_df["clean_tweet"]
y_test = test_df["sentiment"]

# Converting Text to Numbers (TF-IDF)

Models cannot work with text directly.

We use TF-IDF to convert words into numeric features.


TF-IDF gives more importance to words that are meaningful and less importance to very common words.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Training the Model

We use Logistic Regression, which works extremely well with TF-IDF features.

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

# Evaluating the Model

In [7]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8028028028028028
              precision    recall  f1-score   support

  Irrelevant       0.73      0.82      0.77       171
    Negative       0.81      0.82      0.81       266
     Neutral       0.82      0.75      0.79       285
    Positive       0.83      0.83      0.83       277

    accuracy                           0.80       999
   macro avg       0.80      0.81      0.80       999
weighted avg       0.81      0.80      0.80       999



In [8]:
import pickle

# Save model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [11]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import os

# Function to generate next filename
def get_next_filename(base_name="sentiment_report", ext=".pdf"):
    i = 1
    while os.path.exists(f"{base_name}_{i}{ext}"):
        i += 1
    return f"{base_name}_{i}{ext}"

# Generate filename
file_name = get_next_filename()

# Create PDF
doc = SimpleDocTemplate(file_name)
styles = getSampleStyleSheet()

content = []

def add_heading(text):
    content.append(Paragraph(f"<b>{text}</b>", styles["Heading2"]))
    content.append(Spacer(1, 10))

def add_text(text):
    content.append(Paragraph(text, styles["BodyText"]))
    content.append(Spacer(1, 10))

# Title
content.append(Paragraph("<b>Twitter Sentiment Analysis Report</b>", styles["Title"]))
content.append(Spacer(1, 20))

# Content
add_heading("Introduction")
add_text("This project builds a sentiment analysis model using Twitter data.")

add_heading("Model")
add_text("TF-IDF + Logistic Regression was used for classification.")

add_heading("Conclusion")
add_text("The system successfully predicts sentiment from tweets.")

# Build PDF
doc.build(content)

print(f"Report saved as: {file_name}")

Report saved as: sentiment_report_1.pdf
